<a href="https://colab.research.google.com/github/Asuskf/from-nlp-to-agents/blob/embeddings/embeddings/Build_Semantic_Embeddings/Build_Semantic_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Understanding the Geometry Behind LLM Embeddings

Modern Large Language Models (LLMs) represent language as vectors inside high-dimensional spaces known as **embeddings**.

Before working with embeddings containing hundreds or thousands of learned dimensions, it is useful to understand the mathematical principles behind them using a simple, interpretable semantic space.

In this lab you will:

- Build a small semantic embedding space from scratch.
- Measure semantic similarity using cosine similarity.
- Search for the closest semantic neighbor.
- Perform vector arithmetic to solve word analogies.
- Understand why these ideas scale to modern embedding models and Retrieval-Augmented Generation (RAG) systems.

> **Learning Goal:** Understand how linear algebra becomes semantic reasoning.

# Step 1 — Representing Meaning as Vectors

Computers do not understand words directly.

Instead, each word is represented by a numerical vector called an **embedding**.

Real embedding models use hundreds or thousands of dimensions learned automatically during training.

For educational purposes, we'll build a small **4-dimensional semantic space** where every dimension has an intuitive meaning.

| Dimension | Meaning |
|-----------|---------|
| 0 | Royalty |
| 1 | Masculinity |
| 2 | Femininity |
| 3 | Age |

Although modern neural networks discover these dimensions automatically, manually defining them allows us to visualize how semantic reasoning emerges from linear algebra.

In [2]:
import numpy as np

# 4D Semantic Space
# [Royalty, Masculinity, Femininity, Age]

vocabulary = {
    "king": np.array([0.95, 0.90, 0.05, 0.70]),
    "queen": np.array([0.95, 0.05, 0.95, 0.70]),
    "man": np.array([0.05, 0.95, 0.05, 0.60]),
    "woman": np.array([0.05, 0.05, 0.95, 0.60]),
    "apple": np.array([0.00, 0.00, 0.00, 0.00])
}

for word, vector in vocabulary.items():
    print(f"{word:<6} → {vector}")

king   → [0.95 0.9  0.05 0.7 ]
queen  → [0.95 0.05 0.95 0.7 ]
man    → [0.05 0.95 0.05 0.6 ]
woman  → [0.05 0.05 0.95 0.6 ]
apple  → [0. 0. 0. 0.]


Notice that **King** and **Queen** have very similar vectors.

The primary difference lies in the gender-related dimensions.

This property is what makes semantic vector arithmetic possible.

# Step 2 — Measuring Semantic Similarity

How can we determine whether two words have similar meanings?

Instead of measuring the distance between vectors, NLP models compare their **direction**.

This is done using **Cosine Similarity**.

A cosine similarity of:

- **1** → nearly identical meaning
- **0** → unrelated concepts
- **-1** → opposite directions

In [3]:
def cosine_similarity(vec_a, vec_b):

    dot_product = np.dot(vec_a, vec_b)

    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

In [4]:
similarity = cosine_similarity(
    vocabulary["king"],
    vocabulary["queen"]
)

print(f"Similarity(King, Queen): {similarity:.4f}")

Similarity(King, Queen): 0.6598


Even though **King** and **Queen** are not identical vectors, they point in nearly the same direction.

This indicates that they share most of their semantic properties.


# Step 3 — Finding the Closest Semantic Neighbor

Suppose we create a new vector that does not exactly match any word in our vocabulary.

How can we identify which concept it most closely represents?

We'll compare the target vector against every word in the vocabulary using cosine similarity and return the closest match.

In [5]:
def find_closest_word(target_vector, vocabulary, exclude_words=None):

    if exclude_words is None:
        exclude_words = []

    best_word = None
    highest_similarity = -1

    for word, vector in vocabulary.items():

        if word in exclude_words:
            continue

        similarity = cosine_similarity(target_vector, vector)

        print(f"{word:<6} -> {similarity:.4f}")

        if similarity > highest_similarity:
            highest_similarity = similarity
            best_word = word

    return best_word, highest_similarity

In [6]:
closest_word, score = find_closest_word(
    target_vector=vocabulary["king"],
    vocabulary=vocabulary,
    exclude_words=["king"]
)

print("\nClosest semantic neighbor:")
print(f"{closest_word} ({score:.4f})")

queen  -> 0.6598
man    -> 0.7926
woman  -> 0.3350
apple  -> 0.0000

Closest semantic neighbor:
man (0.7926)


# Step 4 — Semantic Vector Arithmetic

One of the most remarkable discoveries in NLP is that embeddings support algebraic operations.

Consider the expression:

**King − Man + Woman**

Conceptually, we are saying:

- Remove masculinity.
- Add femininity.
- Preserve royalty.

If semantic information is truly encoded inside vectors, the resulting point should lie close to **Queen**.

In [7]:
result_vector = (
    vocabulary["king"]
    - vocabulary["man"]
    + vocabulary["woman"]
)

print(result_vector)

[9.5000000e-01 6.9388939e-17 9.5000000e-01 7.0000000e-01]


The resulting vector does not exactly equal the **Queen** vector.

Instead, it lands nearby within the semantic space.

Let's identify the closest known word.

In [8]:
closest_word, score = find_closest_word(
    target_vector=result_vector,
    vocabulary=vocabulary,
    exclude_words=["king", "man", "woman"]
)

print("\n===== RESULT =====")
print(f"King - Man + Woman = {closest_word}")
print(f"Similarity Score = {score:.4f}")

queen  -> 0.9995
apple  -> 0.0000

===== RESULT =====
King - Man + Woman = queen
Similarity Score = 0.9995


The nearest neighbor is **Queen**, demonstrating that semantic concepts can be manipulated using simple vector arithmetic.

This experiment popularized the idea that meaning can emerge from geometry.

# Why This Matters for Modern LLMs

Our semantic space contains only four manually designed dimensions.

Modern embedding models—including Word2Vec, FastText, OpenAI Embeddings, Gemini Embeddings, and many Retrieval-Augmented Generation (RAG) systems—learn hundreds or thousands of latent dimensions automatically.

Despite the increased complexity, the underlying principles remain unchanged:

- Words become vectors.
- Similar meanings occupy nearby regions.
- Cosine similarity measures semantic closeness.
- Linear algebra enables semantic reasoning.

Understanding this small example provides the intuition behind semantic search, recommendation systems, clustering, vector databases, and modern LLM retrieval pipelines.